<a href="https://colab.research.google.com/github/marcelovolta/sr_uba_2026/blob/main/notebooks%20profesor/11_Dos_torres_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Claude fix: already installed in the venv. Running pip here upgrades keras/jax/tensorflow under an
# already-running kernel, which mixes old and new modules (AttributeError: 'keras.src.backend' has no attribute 'numpy').
# If you do need to install, run this cell once and then restart the kernel before running the rest.
# ! pip install -qU keras-rs
# ! pip install jax
# ! pip install jaxlib
# ! pip install -qU tensorflow

### Claude explanation (not from the professor)
Selects **JAX** as the Keras backend. This must be set before `import keras`. TensorFlow is used only for `tf.data`.

In [5]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras_rs
import keras
import tensorflow as tf
import sqlite3
import pandas as pd

# Datos

### Claude explanation (not from the professor)
Loads every `(reader, book, rating)` interaction.

In [6]:
conn = sqlite3.connect('datos/datos.db')

df_ratings = pd.read_sql("SELECT id_lector, id_libro, rating FROM interacciones", conn)
df_ratings.head()

,id_lector,id_libro,rating
0,bioaqua,las-puertas-de-anubis,4
1,bioaqua,antonio-b-el-ruso-ciudadano-de-tercera,10
2,bioaqua,la-momia,10
3,bioaqua,el-clan-del-oso-cavernario-los-hijos-de-la-tie...,10
4,bioaqua,dime-quien-soy-1,10


### Claude explanation (not from the professor)
Builds dictionaries that translate string IDs to consecutive integers and back. Embedding layers need integer indices.

In [7]:
# diccionarios de traducción
unique_user_id = df_ratings["id_lector"].unique().tolist()
user_id_2_id_mapping = {t: i for i, t in enumerate(unique_user_id)}
id_2_user_user_mapping = {user_id_2_id_mapping[k]: k for k in user_id_2_id_mapping}

unique_item_id = df_ratings["id_libro"].unique().tolist()
item_id_2_id_mapping = {t: i for i, t in enumerate(unique_item_id)}
id_2_item_id_mapping = {item_id_2_id_mapping[k]: k for k in item_id_2_id_mapping}

### Claude explanation (not from the professor)
Replaces the string IDs in the DataFrame with their integer index.

In [8]:
df_ratings["id_lector"] = df_ratings["id_lector"].apply(lambda x: user_id_2_id_mapping[x])
df_ratings["id_libro"] = df_ratings["id_libro"].apply(lambda x: item_id_2_id_mapping[x])

df_ratings.head()

,id_lector,id_libro,rating
0,0,0,4
1,0,1,10
2,0,2,10
3,0,3,10
4,0,4,10


### Claude explanation (not from the professor)
Builds the dataset:
- Inputs: reader and book indices.
- Target: the rating rescaled from 1–10 to 0–1, to match the model's sigmoid output.
- Shuffled once with a fixed seed, then split 80/20.

**Note:** `skip(df_ratings.shape[0] - cut)` skips only 20% of the rows, so the test set is the *last 80%* and mostly overlaps training. It should be `skip(cut)`.

In [9]:
dataset_inputs = {
    "user_id": df_ratings['id_lector'].to_numpy(dtype="float32"),
    'item_id': df_ratings['id_libro'].to_numpy(dtype="float32")
}

# dataset_outputs = (df_ratings['rating'].to_numpy() - 1.0) / 9.0
dataset_outputs = ((df_ratings['rating'].to_numpy() - 1.0) / 9.0).astype("float32")  # Claude fix: JAX uses float32; avoids the float64 truncation warning

dataset = tf.data.Dataset.from_tensor_slices((dataset_inputs, dataset_outputs))
dataset = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

cut = int(df_ratings.shape[0] * 0.8)
train_dataset = dataset.take(cut).batch(256).cache()
test_dataset = dataset.skip(df_ratings.shape[0] - cut).batch(256).cache()

## Modelo

### Claude explanation (not from the professor)
**Ranking model.** One embedding table for readers and one for books (512 dimensions each). The two vectors are concatenated and passed through dense layers ending in one sigmoid unit, the predicted (rescaled) rating. Unlike a plain dot product, the dense layers can learn non-linear reader–book interactions.

In [10]:
class RankingModel(keras.Model):
    """Crea un modelo de ranking

    Args:
      num_users: Number of entries in the user embedding table.
      num_items: Number of entries in the candidate embedding table.
      embedding_dimension: Output dimension for user and movie embedding tables.
    """

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dimension=512, # embedding_dimension es hiperparámetro
        **kwargs,
    ):
        super().__init__(**kwargs)

        # Embedding de los usuarios
        self.user_embedding = keras.layers.Embedding(num_users, embedding_dimension)

        # Embedding de los items
        self.item_embedding = keras.layers.Embedding(num_items, embedding_dimension)

        # Predicciones
        self.ratings = keras.Sequential(
            [
                # Capas densas
                keras.layers.Dense(256, activation="relu"),
                keras.layers.Dense(64, activation="relu"),

                # Predicciones en la capa final
                keras.layers.Dense(1, activation="sigmoid"),
            ]
        )

    def call(self, inputs):
        user_id = inputs["user_id"]
        item_id = inputs["item_id"]

        user_embeddings = self.user_embedding(user_id)
        item_embeddings = self.item_embedding(item_id)
        return self.ratings(keras.ops.concatenate([user_embeddings, item_embeddings], axis=1))

## Entrenamiento y evaluación

### Claude explanation (not from the professor)
Creates the model (`+1` leaves room for one extra index) and compiles it with MSE loss, RMSE as the metric and the Adagrad optimizer.

In [11]:
ranking_model = RankingModel(len(unique_user_id) + 1, len(unique_item_id) + 1)
ranking_model.compile(
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.RootMeanSquaredError()],
    optimizer=keras.optimizers.Adagrad(learning_rate=0.1),
)

### Claude explanation (not from the professor)
Trains for 10 epochs and evaluates on the test set every 5. `batch_size` is ignored because the datasets are already batched.

In [12]:
history = ranking_model.fit(train_dataset, validation_data=test_dataset, validation_freq=5, epochs=10, batch_size=256)

/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/jax/_src/core.py:1976: UserWarning: Explicitly requested dtype float64 requested in ShapeDtypeStruct is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return aval_fn(x)


AttributeError: module 'keras.src.backend' has no attribute 'numpy'

## Predicción

### Claude explanation (not from the professor)
Scratch cell for looking up a reader's or book's integer index.

In [ ]:
#user_id_2_id_mapping["popocito"]
#item_id_2_id_mapping["la-catedral-del-mar"]

123

### Claude explanation (not from the professor)
Predicts the rating `popocito` would give four specific books, then maps the output from 0–1 back to the 1–10 scale (`× 9 + 1`).

In [ ]:
user_id = user_id_2_id_mapping["popocito"]
item_ids = [item_id_2_id_mapping[i] for i in ["1984", "el-hobbit", "el-silmarillion", "la-catedral-del-mar"]]

predictions = ranking_model.predict({
    "user_id": keras.ops.array([user_id] * len(item_ids)),
    "item_id": keras.ops.array(item_ids),
})

predictions = keras.ops.convert_to_numpy(keras.ops.squeeze(predictions, axis=1))

for id_libro, prediction in zip(item_ids, predictions):
    print(f"{id_2_item_id_mapping[id_libro]} --> {prediction * 9.0 + 1.0:,.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
1984 --> 7.54
el-hobbit --> 7.27
el-silmarillion --> 6.15
la-catedral-del-mar --> 6.43


### Claude explanation (not from the professor)
Scores every book for one reader, as a step toward ranking them and recommending the best.

**Note:** `42` is an *integer index*, but the SQL compares it with the string `id_lector` column, so nothing is excluded and the reader's already-rated books are scored too. Also, `if item_id_2_id_mapping.get(u)` drops the book whose index is 0.

In [ ]:
user_id = 42

df_item_ids = pd.read_sql("SELECT id_libro FROM libros WHERE id_libro NOT IN (SELECT id_libro FROM interacciones WHERE id_lector = ?)", conn, params=[user_id])
item_ids = [item_id_2_id_mapping[u] for u in df_item_ids["id_libro"].to_list() if item_id_2_id_mapping.get(u)]

print(len(item_ids))

predictions = ranking_model.predict({
    "user_id": keras.ops.array([user_id] * len(item_ids)),
    "item_id": keras.ops.array(item_ids),
})

48066
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
